In [ ]:
install.packages("vctrs")
install.packages("dplyr")
install.packages("data.table")
install.packages("VectorForgeML")
library(VectorForgeML)
library(data.table)
library(dplyr)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)


Attaching package: ‘dplyr’


The following objects are masked from ‘package:data.table’:

    between, first, last


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union




In [ ]:
system("pip install -U gdown")

system("gdown --fuzzy 'https://drive.google.com/file/d/1sJRzoxDv5bn8jg__s9C1y83obpHt-oqV/view?usp=sharing' -O data.csv")
list.files()
data <- fread("data.csv")

In [ ]:
head(data)

Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,⋯,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<dbl>,<dbl>,⋯,<int>,<dbl>,<dbl>,<int>,<int>,<dbl>,<dbl>,<int>,<int>,<chr>
54865,3,2,0,12,0,6,6,6,0,⋯,20,0,0,0,0,0,0,0,0,BENIGN
55054,109,1,1,6,6,6,6,6,0,⋯,20,0,0,0,0,0,0,0,0,BENIGN
55055,52,1,1,6,6,6,6,6,0,⋯,20,0,0,0,0,0,0,0,0,BENIGN
46236,34,1,1,6,6,6,6,6,0,⋯,20,0,0,0,0,0,0,0,0,BENIGN
54863,3,2,0,12,0,6,6,6,0,⋯,20,0,0,0,0,0,0,0,0,BENIGN
54871,1022,2,0,12,0,6,6,6,0,⋯,20,0,0,0,0,0,0,0,0,BENIGN


In [ ]:
dim(data)
nrow(data)
ncol(data)

[1] 2827876      79

[1] 2827876

[1] 79

In [ ]:

names(data) <- trimws(names(data))


# Convert label properly
y <- as.factor(data$Label)
y <- as.numeric(y) - 1

X <- data
X$Label <- NULL

# Replace infinite values
numeric_cols <- names(X)[sapply(X, is.numeric)]

X[, (numeric_cols) := lapply(.SD, function(col) {
  col[is.infinite(col)] <- NA
  col
}), .SDcols = numeric_cols]

# Remove NA rows safely
data <- cbind(X, Label=y)
data <- na.omit(data)

y <- data$Label
X <- data
X$Label <- NULL

In [ ]:
ls("package:VectorForgeML")

[1] "accuracy_score"        "ColumnTransformer"     "confusion_matrix"     
 [4] "confusion_stats"       "DecisionTree"          "drop_constant_columns"
 [7] "f1_score"              "find_best_k"           "fit_linear_model"     
[10] "KMeans"                "KNN"                   "LabelEncoder"         
[13] "LinearRegression"      "LogisticRegression"    "macro_f1"             
[16] "macro_precision"       "macro_recall"          "MinMaxScaler"         
[19] "mse"                   "OneHotEncoder"         "PCA"                  
[22] "Pipeline"              "plot_confusion_matrix" "precision_score"      
[25] "predict_linear_model"  "r2_score"              "RandomForest"         
[28] "recall_score"          "RidgeRegression"       "rmse"                 
[31] "SoftmaxRegression"     "StandardScaler"        "train_test_split"

In [ ]:
# Identify columns
cat_cols <- names(X)[sapply(X, is.character)]
num_cols <- names(X)[!sapply(X, is.character)]

# Convert categorical → factor (FIXED)
X[, (cat_cols) := lapply(.SD, as.factor), .SDcols = cat_cols]

# Split
split <- train_test_split(X, y, 0.2, 42)

# Preprocessing
pre <- ColumnTransformer$new(
  num_cols, cat_cols,
  StandardScaler$new(),
  OneHotEncoder$new()
)

# Model
pipe <- Pipeline$new(list(
  pre,
  RandomForest$new(ntrees=1, max_depth=7, 4, mode="classification")
))

# Train
pipe$fit(split$X_train, split$y_train)

# Predict
pred <- pipe$predict(split$X_test)


cat("Accuracy:", accuracy_score(split$y_test, pred), "\n")

Accuracy: 0.9754657 


In [ ]:
table(y)
prop.table(table(y))

y
      0       1       2       3       4       5       6       7       8       9 
2271320    1956  128025   10293  230124    5499    5796    7935      11      36 
     10      11      12      13      14 
 158804    5897    1507      21     652 

y
           0            1            2            3            4            5 
8.031894e-01 6.916852e-04 4.527249e-02 3.639834e-03 8.137698e-02 1.944569e-03 
           6            7            8            9           10           11 
2.049595e-03 2.805993e-03 3.889845e-06 1.273040e-05 5.615663e-02 2.085311e-03 
          12           13           14 
5.329088e-04 7.426068e-06 2.305617e-04 

In [ ]:
conf <- confusion_matrix(y, split$y_test)

conf

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14
0,224091,208,12616,1043,22542,558,577,772,1,3,15553,571,146,2,62
1,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,102824,76,5765,460,10537,262,285,357,0,2,7100,274,57,1,25
3,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
5,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
6,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
7,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
8,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
9,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
